# Data Cleaning

In [70]:
import pandas as pd 
import os

#### Customers

In [71]:
customers= pd.read_json(r'D:\Ahmed\Data engineering - MSC\Banking-ETL-Pipeline-Data-Processing\Data\customers.json')
customers


,CustomerID,FirstName,LastName,Phone,Email,Address,JoinDate
0,1,Dustin,Diaz,(107)429-6508,jenniferbaker@montgomery-cardenas.com,"198 Elizabeth Bypass, Allenmouth, IN 60876",1575763200000
1,2,Jessica,Anderson,+1-936-234-0473x4335,vanessamoses@velasquez.com,"PSC 6146, Box 0055, APO AE 83600",1456963200000
2,3,Jeremy,Wagner,593-321-1624x10444,johntrujillo@golden.com,"636 John Oval Apt. 207, Angelfurt, HI 24913",1687219200000
3,4,Crystal,Roberts,502-211-0360x097,stewartsean@gmail.com,"8398 Carter Views, North Austinhaven, RI 17496",1587081600000
4,5,Anna,Bryant,776-228-1583x2038,jacobcantrell@gmail.com,"52819 Craig Springs Apt. 576, Steventown, MD 2...",1742688000000
...,...,...,...,...,...,...,...
4995,4996,Toni,Rivera,+1-931-263-9930x318,phelpsvanessa@gmail.com,"80759 Brennan Pines Suite 917, Ashleyberg, CO ...",1468108800000
4996,4997,Mark,Vasquez,(771)739-2512,stoneamanda@king.com,"389 Christopher Mount Suite 748, Stevenston, A...",1624060800000
4997,4998,Alyssa,Becker,(126)870-8435x9585,mreyes@yahoo.com,"4444 Smith Villages, Lake Gilbertbury, ID 13118",1467072000000
4998,4999,Mary,Torres,(584)018-9295x7963,psexton@gmail.com,"3491 Marc Freeway Apt. 884, Lopezton, MS 61950",1487376000000


In [72]:
customers.head()

,CustomerID,FirstName,LastName,Phone,Email,Address,JoinDate
0,1,Dustin,Diaz,(107)429-6508,jenniferbaker@montgomery-cardenas.com,"198 Elizabeth Bypass, Allenmouth, IN 60876",1575763200000
1,2,Jessica,Anderson,+1-936-234-0473x4335,vanessamoses@velasquez.com,"PSC 6146, Box 0055, APO AE 83600",1456963200000
2,3,Jeremy,Wagner,593-321-1624x10444,johntrujillo@golden.com,"636 John Oval Apt. 207, Angelfurt, HI 24913",1687219200000
3,4,Crystal,Roberts,502-211-0360x097,stewartsean@gmail.com,"8398 Carter Views, North Austinhaven, RI 17496",1587081600000
4,5,Anna,Bryant,776-228-1583x2038,jacobcantrell@gmail.com,"52819 Craig Springs Apt. 576, Steventown, MD 2...",1742688000000


In [98]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   CustomerID  5000 non-null   int64 
 1   FirstName   5000 non-null   str   
 2   LastName    5000 non-null   str   
 3   Phone       5000 non-null   str   
 4   Email       5000 non-null   str   
 5   Address     5000 non-null   str   
 6   JoinDate    5000 non-null   object
dtypes: int64(1), object(1), str(5)
memory usage: 273.6+ KB


In [74]:
def check_missing_values(df, name="DataFrame"):
    """
    Check missing values in a pandas DataFrame.
 
    Parameters:
        df   : pd.DataFrame
        name : str — label shown in the report header
 
    Returns:
        pd.DataFrame — summary table with one row per column
    """
    total_rows = len(df)
 
    missing_count = df.isnull().sum()
    missing_pct   = (missing_count / total_rows * 100).round(2)
    present_count = total_rows - missing_count
 
    summary = pd.DataFrame({
        "column"       : df.columns,
        "dtype"        : df.dtypes.values,
        "total_rows"   : total_rows,
        "present"      : present_count.values,
        "missing"      : missing_count.values,
        "missing_%"    : missing_pct.values,
    })
 
    total_missing = missing_count.sum()
    total_cells   = total_rows * len(df.columns)
 
    print(f"\n{'='*55}")
    print(f"  Missing Value Report — {name}")
    print(f"{'='*55}")
    print(f"  Rows        : {total_rows:,}")
    print(f"  Columns     : {len(df.columns)}")
    print(f"  Total cells : {total_cells:,}")
    print(f"  Missing     : {total_missing:,}  ({round(total_missing / total_cells * 100, 2)}%)")
    print(f"{'='*55}\n")
    print(summary.to_string(index=False))
    print()
 
    return summary

In [75]:
check_missing_values(customers)


  Missing Value Report — DataFrame
  Rows        : 5,000
  Columns     : 7
  Total cells : 35,000
  Missing     : 0  (0.0%)

    column dtype  total_rows  present  missing  missing_%
CustomerID int64        5000     5000        0        0.0
 FirstName   str        5000     5000        0        0.0
  LastName   str        5000     5000        0        0.0
     Phone   str        5000     5000        0        0.0
     Email   str        5000     5000        0        0.0
   Address   str        5000     5000        0        0.0
  JoinDate int64        5000     5000        0        0.0



,column,dtype,total_rows,present,missing,missing_%
0,CustomerID,int64,5000,5000,0,0.0
1,FirstName,str,5000,5000,0,0.0
2,LastName,str,5000,5000,0,0.0
3,Phone,str,5000,5000,0,0.0
4,Email,str,5000,5000,0,0.0
5,Address,str,5000,5000,0,0.0
6,JoinDate,int64,5000,5000,0,0.0


In [76]:
def check_duplicates(df, name="DataFrame", pk_col=None, subset_cols=None):
    """
    Detect duplicates in a DataFrame.
 
    Parameters:
        df          : pd.DataFrame
        name        : str   — label shown in the report
        pk_col      : str   — primary-key column to check for PK duplicates
        subset_cols : list  — extra columns to check for business-key duplicates
                              (e.g. ["Email"] for customers)
 
    Returns:
        dict with keys:
            full_row_dups   — DataFrame of full-row duplicate rows
            pk_dups         — DataFrame of PK duplicate rows (if pk_col given)
            subset_dups     — dict of {col: DataFrame} for each subset_col
    """
    results = {}
 
    total_rows = len(df)
 
    print(f"\n{'='*55}")
    print(f"  Duplicate Report — {name}")
    print(f"{'='*55}")
    print(f"  Rows : {total_rows:,}")
 
    # 1. Full-row duplicates
    full_mask = df.duplicated(keep=False)
    full_dups = df[full_mask]
    results["full_row_dups"] = full_dups
    count = df.duplicated().sum()
    status = "✓ None" if count == 0 else f"⚠  {count} duplicate rows found"
    print(f"\n  Full-row duplicates      : {status}")
    if count > 0:
        print(full_dups.to_string(index=False))
 
    # 2. Primary-key duplicates
    if pk_col:
        pk_mask = df.duplicated(subset=[pk_col], keep=False)
        pk_dups = df[pk_mask]
        results["pk_dups"] = pk_dups
        pk_count = df.duplicated(subset=[pk_col]).sum()
        status = "✓ None" if pk_count == 0 else f"⚠  {pk_count} duplicate PKs found"
        print(f"  PK duplicates ({pk_col:<12}): {status}")
        if pk_count > 0:
            print(pk_dups.to_string(index=False))
 
    # 3. Business-key / subset duplicates
    subset_dups = {}
    if subset_cols:
        for col in subset_cols:
            mask  = df.duplicated(subset=[col], keep=False)
            dups  = df[mask].sort_values(col)
            count = df.duplicated(subset=[col]).sum()
            subset_dups[col] = dups
            status = "✓ None" if count == 0 else f"⚠  {count} duplicate values found"
            print(f"  Duplicate {col:<20}: {status}")
            if count > 0:
                print(dups.to_string(index=False))
 
    results["subset_dups"] = subset_dups
    print()
    return results

In [77]:
check_duplicates(customers)


  Duplicate Report — DataFrame
  Rows : 5,000

  Full-row duplicates      : ✓ None



{'full_row_dups': Empty DataFrame
 Columns: [CustomerID, FirstName, LastName, Phone, Email, Address, JoinDate]
 Index: [],
 'subset_dups': {}}

In [78]:
customers['JoinDate']=pd.to_datetime(customers['JoinDate'],unit='ms').dt.date
customers.head()

,CustomerID,FirstName,LastName,Phone,Email,Address,JoinDate
0,1,Dustin,Diaz,(107)429-6508,jenniferbaker@montgomery-cardenas.com,"198 Elizabeth Bypass, Allenmouth, IN 60876",2019-12-08
1,2,Jessica,Anderson,+1-936-234-0473x4335,vanessamoses@velasquez.com,"PSC 6146, Box 0055, APO AE 83600",2016-03-03
2,3,Jeremy,Wagner,593-321-1624x10444,johntrujillo@golden.com,"636 John Oval Apt. 207, Angelfurt, HI 24913",2023-06-20
3,4,Crystal,Roberts,502-211-0360x097,stewartsean@gmail.com,"8398 Carter Views, North Austinhaven, RI 17496",2020-04-17
4,5,Anna,Bryant,776-228-1583x2038,jacobcantrell@gmail.com,"52819 Craig Springs Apt. 576, Steventown, MD 2...",2025-03-23


In [79]:
##Seprating Dataset
import pandas as pd

file_path = r"D:\Ahmed\Data engineering - MSC\Banking-ETL-Pipeline-Data-Processing\Data\Banking_Analytics_Dataset.xlsx"

# Load all sheets
sheets = pd.read_excel(file_path, sheet_name=None)

# Save each sheet as a separate CSV
for sheet_name, df in sheets.items():
    # Clean sheet name (optional but recommended)
    safe_name = sheet_name.replace(" ", "_").replace("/", "_")
    
    output_file = f"{safe_name}.csv"
    df.to_csv(output_file, index=False)
    
    print(f"Saved: {output_file}")

Saved: Accounts.csv
Saved: Transactions.csv
Saved: Loans.csv
Saved: Cards.csv
Saved: SupportCalls.csv


### Accounts

In [86]:
Accounts=pd.read_csv(r'D:\Ahmed\Data engineering - MSC\Banking-ETL-Pipeline-Data-Processing\Data\Accounts.csv')
Accounts.head()

,AccountID,CustomerID,AccountType,Balance,CreatedDate
0,1,1828,Savings,53911,2019-06-12
1,2,3740,Business,39910,2015-07-14
2,3,1247,Savings,75903,2018-03-23
3,4,3843,Business,47519,2024-12-14
4,5,4693,Savings,34233,2019-07-26


In [87]:
Accounts.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   AccountID    5000 non-null   int64
 1   CustomerID   5000 non-null   int64
 2   AccountType  5000 non-null   str  
 3   Balance      5000 non-null   int64
 4   CreatedDate  5000 non-null   str  
dtypes: int64(3), str(2)
memory usage: 195.4 KB


In [95]:
check_missing_values(Accounts)


  Missing Value Report — DataFrame
  Rows        : 5,000
  Columns     : 5
  Total cells : 25,000
  Missing     : 0  (0.0%)

     column dtype  total_rows  present  missing  missing_%
  AccountID int64        5000     5000        0        0.0
 CustomerID int64        5000     5000        0        0.0
AccountType   str        5000     5000        0        0.0
    Balance int64        5000     5000        0        0.0
CreatedDate   str        5000     5000        0        0.0



,column,dtype,total_rows,present,missing,missing_%
0,AccountID,int64,5000,5000,0,0.0
1,CustomerID,int64,5000,5000,0,0.0
2,AccountType,str,5000,5000,0,0.0
3,Balance,int64,5000,5000,0,0.0
4,CreatedDate,str,5000,5000,0,0.0


In [96]:
check_duplicates(Accounts)


  Duplicate Report — DataFrame
  Rows : 5,000

  Full-row duplicates      : ✓ None



{'full_row_dups': Empty DataFrame
 Columns: [AccountID, CustomerID, AccountType, Balance, CreatedDate]
 Index: [],
 'subset_dups': {}}

In [91]:
Accounts['AccountType'].unique()

<StringArray>
['Savings', 'Business', 'Checking']
Length: 3, dtype: str

In [93]:
Valid_accounttypes={'Savings', 'Business', 'Checking'}
Accounts=Accounts[
    Accounts['AccountType'].isin(Valid_accounttypes)
]

In [ ]:
#### check balance 
Accounts=Accounts[Accounts['Balance']>=0]

In [97]:
Accounts['CreatedDate']=pd.to_datetime(Accounts['CreatedDate'],unit='ms').dt.date
Accounts.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   AccountID    5000 non-null   int64 
 1   CustomerID   5000 non-null   int64 
 2   AccountType  5000 non-null   str   
 3   Balance      5000 non-null   int64 
 4   CreatedDate  5000 non-null   object
dtypes: int64(3), object(1), str(1)
memory usage: 195.4+ KB


### Transactions

In [80]:
Transactions=pd.read_csv(r'D:\Ahmed\Data engineering - MSC\Banking-ETL-Pipeline-Data-Processing\Data\Transactions.csv')
Transactions.head()

,TransactionID,AccountID,TransactionType,Amount,TransactionDate
0,1,3913,Transfer,3150.12,2023-09-24
1,2,2591,Transfer,6212.12,2022-06-07
2,3,3277,Transfer,451.72,2024-11-24
3,4,3404,Deposit,8525.28,2023-04-06
4,5,4345,Deposit,7306.17,2025-01-21


In [81]:
Transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   TransactionID    20000 non-null  int64  
 1   AccountID        20000 non-null  int64  
 2   TransactionType  20000 non-null  str    
 3   Amount           20000 non-null  float64
 4   TransactionDate  20000 non-null  str    
dtypes: float64(1), int64(2), str(2)
memory usage: 781.4 KB


In [82]:
check_missing_values(Transactions)


  Missing Value Report — DataFrame
  Rows        : 20,000
  Columns     : 5
  Total cells : 100,000
  Missing     : 0  (0.0%)

         column   dtype  total_rows  present  missing  missing_%
  TransactionID   int64       20000    20000        0        0.0
      AccountID   int64       20000    20000        0        0.0
TransactionType     str       20000    20000        0        0.0
         Amount float64       20000    20000        0        0.0
TransactionDate     str       20000    20000        0        0.0



,column,dtype,total_rows,present,missing,missing_%
0,TransactionID,int64,20000,20000,0,0.0
1,AccountID,int64,20000,20000,0,0.0
2,TransactionType,str,20000,20000,0,0.0
3,Amount,float64,20000,20000,0,0.0
4,TransactionDate,str,20000,20000,0,0.0


In [83]:
check_duplicates(Transactions)


  Duplicate Report — DataFrame
  Rows : 20,000

  Full-row duplicates      : ✓ None



{'full_row_dups': Empty DataFrame
 Columns: [TransactionID, AccountID, TransactionType, Amount, TransactionDate]
 Index: [],
 'subset_dups': {}}

In [84]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'ODBC Driver 18 for SQL Server', 'Oracle in OraDB23Home1']


In [85]:
from sqlalchemy import create_engine, text

server='localhost'
database='BANKING_ETL'
username='sa'
password='StrongPass123'

connection_string = (
    f"mssql+pyodbc://{username}:{password}@{server}/{database}"
    "?driver=ODBC+Driver+18+for+SQL+Server&TrustServerCertificate=yes"
)

engine=create_engine(connection_string)

#Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT name FROM sys.databases"))
    for row in result:
        print(row.name)

master
tempdb
model
msdb
DataWarehouse
BikeStores
BANKING_ETL
